# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar320250068-prog/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook regenerates the numbers in the deployed paper, section by section, and closes with the
ML-12 communication cuts. The paper lives at the GitHub Pages URL recorded in
`submission/paper_url.txt`; its source is `docs/index.html`.

> Working with an AI assistant? Read `skills/README.md` first; this task used
> `writing-research-papers` + `deploying-static-pages`.

## 0. Setup (repo-root resolution, local + Colab)

In [1]:
import os, sys, subprocess, json
import numpy as np, pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold, StratifiedKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omar320250068-prog/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    if not os.path.isdir("data/raw"):
        for cand in (r"C:\Users\OMAR HINDAWY\flyrank-ml-internship", os.getcwd()):
            if os.path.isdir(os.path.join(cand, "data", "raw")):
                os.chdir(cand); break

os.makedirs("work/outputs", exist_ok=True)
SEED = 42
print("Working dir:", os.getcwd())
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__, "| numpy:", np.__version__)

Working dir: C:\Users\OMAR HINDAWY\flyrank-ml-internship
scikit-learn: 1.9.1 | pandas: 3.0.6 | numpy: 2.5.4


## 1. Question

*The research question and the decision it supports.* The unit is one content page; the output is a
ranked score; the human action is inspect-and-decide; the cost of a wrong call is a wasted review slot.

In [2]:
# Shared slice + proxy label + leakage-safe feature frame (identical across all notebooks).
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
vis = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
         (df["avg_position"] <= 20)].copy()
vis["pos_tier"] = pd.cut(vis["avg_position"], bins=[0, 3, 10, 20],
                         labels=["top_3", "page_1", "striking"]).astype(str)
tier_median = vis.groupby("pos_tier")["ctr"].transform("median")
vis["ctr_vs_tier"] = vis["ctr"] / tier_median.replace(0, np.nan)
vis["under_capturing"] = ((vis["ctr_vs_tier"] < 0.50) &
                           (vis["impressions_90d"] >= 1000)).astype(int)

num_feats = ["impressions_90d", "avg_position", "content_age_days",
             "days_since_last_update", "sessions_90d", "engagement_rate",
             "scroll_rate", "word_count", "search_volume"]
cat_feats = ["pos_tier", "main_intent", "content_type", "position_tier",
             "age_tier", "freshness_tier"]
num_feats = [c for c in num_feats if c in vis.columns]
cat_feats = [c for c in cat_feats if c in vis.columns]
X = pd.concat([vis[num_feats].apply(pd.to_numeric, errors="coerce").fillna(0),
               pd.get_dummies(vis[cat_feats].astype(str))], axis=1)
y = vis["under_capturing"].values
groups = vis["client_id"].values
print(f"Slice: {len(vis):,} pages | clients: {vis['client_id'].nunique()} | base rate: {y.mean():.3f}")

Slice: 12,023 pages | clients: 28 | base rate: 0.193


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.* Source is the
anonymised starter release (one row per pseudonymised page, 90-day rolling fields). Excluded:
label-defining columns (`clicks_90d`, `ctr`, `ctr_vs_tier`), trend/label-family columns
(`trend_direction`, `trend_pct`, `is_declining_label`), and identifiers.

In [3]:
excluded = ["clicks_90d", "ctr", "ctr_vs_tier", "trend_direction", "trend_pct",
            "is_declining_label", "content_id", "client_id"]
present = [c for c in excluded if c in set(num_feats) | set(cat_feats)]
print("Excluded columns requested as features:", present or "none -> leak-safe")
print("Rows dropped for missing position (avg_position=0):",
      int((df["avg_position"] == 0).sum()), "of", len(df))
print("Slice keeps impressions_90d >= 500 and position in (0, 20].")

Excluded columns requested as features: none -> leak-safe
Rows dropped for missing position (avg_position=0): 1205 of 30000
Slice keeps impressions_90d >= 500 and position in (0, 20].


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.* The label is a
**proxy** (`ctr_vs_tier < 0.50`). Validation is **client-grouped** (`GroupKFold(5)`). Baseline is the
transparent rule `(ctr < 0.5) × log1p(impressions_90d)`.

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def oof(model_fn, splitter, use_groups=True):
    preds = np.zeros(len(y))
    folds = splitter.split(X, y, groups) if use_groups else splitter.split(X, y)
    for tr, te in folds:
        m = model_fn(); m.fit(X.iloc[tr], y[tr])
        preds[te] = m.predict_proba(X.iloc[te])[:, 1]
    return preds

# Transparent baseline rule (no fitting).
rule_score = (vis["ctr"] < 0.5).astype(float).values * np.log1p(vis["impressions_90d"].values)

# Leakage check: adding clicks_90d back should push the score toward 1.0.
X_leaky = pd.concat([X, vis[["clicks_90d"]].fillna(0)], axis=1)
leaky = np.zeros(len(y))
for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_leaky, y):
    m = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced",
                               random_state=SEED, n_jobs=-1)
    m.fit(X_leaky.iloc[tr], y[tr]); leaky[te] = m.predict_proba(X_leaky.iloc[te])[:, 1]
print("Leakage probe — P@50 without clicks_90d:",
      round(precision_at_k(oof(lambda: RandomForestClassifier(n_estimators=200, max_depth=8,
            class_weight='balanced', random_state=SEED, n_jobs=-1),
            StratifiedKFold(5, shuffle=True, random_state=SEED), use_groups=False), y, 50), 3))
print("Leakage probe — P@50 WITH    clicks_90d:", round(precision_at_k(leaky, y, 50), 3),
      "(jump toward 1.0 = confession)")

Leakage probe — P@50 without clicks_90d: 0.9
Leakage probe — P@50 WITH    clicks_90d: 1.0 (jump toward 1.0 = confession)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.* Every method on `GroupKFold(5)` by client,
seed 42.

In [5]:
models = {
    "baseline_rule": None,
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED)),
    "tree_d2": lambda: DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=SEED),
    "tree_d3": lambda: DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED),
    "random_forest": lambda: RandomForestClassifier(n_estimators=300, max_depth=8,
        class_weight="balanced", random_state=SEED, n_jobs=-1),
}
rows = []
for name, fn in models.items():
    p = rule_score if name == "baseline_rule" else oof(fn, GroupKFold(n_splits=5))
    rows.append({"method": name, "P@20": round(precision_at_k(p, y, 20), 3),
                 "P@50": round(precision_at_k(p, y, 50), 3),
                 "P@100": round(precision_at_k(p, y, 100), 3),
                 "ROC_AUC": round(roc_auc_score(y, p), 3),
                 "AP": round(average_precision_score(y, p), 3)})
table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))
print(f"base_rate {y.mean():.3f} | n {len(y)} | clients {vis['client_id'].nunique()}")

# C2: the split gap (same model, only the split changes).
rf_rand = oof(models["random_forest"], StratifiedKFold(5, shuffle=True, random_state=SEED), use_groups=False)
rf_grp = oof(models["random_forest"], GroupKFold(n_splits=5))
print(f"\nC2 — RF P@50 random split: {precision_at_k(rf_rand, y, 50):.3f} | "
      f"grouped split: {precision_at_k(rf_grp, y, 50):.3f}")

       method  P@20  P@50  P@100  ROC_AUC    AP
baseline_rule  0.30  0.22   0.24    0.653 0.246
       logreg  0.65  0.58   0.56    0.607 0.265
      tree_d2  0.45  0.50   0.49    0.712 0.328
      tree_d3  0.65  0.66   0.63    0.744 0.404
random_forest  0.45  0.60   0.64    0.777 0.436
base_rate 0.193 | n 12023 | clients 28



C2 — RF P@50 random split: 0.920 | grouped split: 0.600


In [6]:
# Commit the paper's core receipt so its Table 1 traces back to a regenerated file.
receipt = {
    "notebook": "capstone",
    "split": "GroupKFold(5) by client_id, seed 42",
    "n": int(len(y)), "n_clients": int(vis["client_id"].nunique()),
    "base_rate": round(float(y.mean()), 3),
    "label": "under_capturing = (ctr_vs_tier < 0.50) & (impressions_90d >= 1000)",
    "results": table1.to_dict(orient="records"),
    "c2_split_gap": {"rf_random_P@50": round(precision_at_k(rf_rand, y, 50), 3),
                     "rf_grouped_P@50": round(precision_at_k(rf_grp, y, 50), 3)},
}
with open("work/outputs/capstone_core_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2)
print("Wrote work/outputs/capstone_core_metrics.json")

Wrote work/outputs/capstone_core_metrics.json


## 5. Limitations

*What this work cannot claim.* Proxy label (circular with CTR), no causal claim, one snapshot, modest
absolute precision, measurement traps in zero engagement/scroll, survivorship from the ≥500-impressions
floor.

In [7]:
# Quantify two limitations so the paper's limitations section is backed by numbers.
n_zero_eng = int(((vis["engagement_rate"].fillna(0) == 0)).sum())
print(f"Pages with engagement_rate == 0 (often missing instrumentation): {n_zero_eng:,} "
      f"({n_zero_eng/len(vis):.1%} of queue)")
vis["age_bucket"] = pd.cut(vis["content_age_days"], bins=[0, 90, 180, 270, 365, 10**6],
                           labels=["0-90", "91-180", "181-270", "271-365", "365+"])
decay = vis.groupby("age_bucket", observed=True)["under_capturing"].mean()
print("\nUnder-capture rate by age bucket (observed, directional):")
print(decay.round(3).to_string())

Pages with engagement_rate == 0 (often missing instrumentation): 6,324 (52.6% of queue)

Under-capture rate by age bucket (observed, directional):
age_bucket
0-90       0.028
91-180     0.176
181-270    0.211
271-365    0.181
365+       0.235


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.* The model orders; a human decides.
Reason codes annotate; the no-go list forbids automation.

In [8]:
import json as _json
play = _json.load(open("work/outputs/w07_playbook_metrics.json"))
print("Queue rows:", play["queue_rows"], "| base rate:", play["base_rate"])
print("Out-of-fold precision@20:", play["oof_precision_at_20"],
      "| @50:", play["oof_precision_at_50"])
print("\nReason-code counts:")
for k, v in sorted(play["reason_code_counts"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:28s} {v}")
print("\nNo-go: no auto-edit/publish/refresh/delete; no client decisions; no production pipeline.")

Queue rows: 12023 | base rate: 0.193
Out-of-fold precision@20: 0.75 | @50: 0.72

Reason-code counts:
  engagement_gap_scan          6359
  below_par_capture            3479
  striking_distance_push       1753
  high_volume_low_ctr          384
  thin_content_low_capture     48

No-go: no auto-edit/publish/refresh/delete; no client decisions; no production pipeline.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.* Figures live in `docs/img/` and
are referenced by `docs/index.html` with relative paths.

In [9]:
for p in ["docs/img/fig1_precision_at_k.png", "docs/img/fig2_split_gap.png",
          "docs/img/fig3_decay.png", "docs/img/fig4_reason_codes.png"]:
    print(("FOUND  " if os.path.exists(p) else "MISSING") + " " + p)
print("\nPaper source: docs/index.html  |  deployed URL: submission/paper_url.txt")

FOUND   docs/img/fig1_precision_at_k.png
FOUND   docs/img/fig2_split_gap.png
FOUND   docs/img/fig3_decay.png
FOUND   docs/img/fig4_reason_codes.png

Paper source: docs/index.html  |  deployed URL: submission/paper_url.txt


## 8. ML-12 communication cuts

The same work, three readers — evidence and numbers never change, only framing.

### 8a. Five-minute demo outline

1. **The decision (30s).** A team can review ~50 pages a cycle; the question is *which first*.
2. **The data (30s).** 30,000 pseudonymised pages → a 12,023-page decision slice, 28 clients.
3. **The trap (60s).** A random split says 0.92 precision@50; grouping by client says 0.60. Same
   model — the gap is memorisation.
4. **The result (60s).** Random forest leads the rule (P@50 0.60 vs 0.22; base rate 0.19), and it is
   decision-support, not causal.
5. **The output (60s).** A ranked, reason-coded queue with a human-review gate and a no-go list.
6. **Limits & next (60s).** Proxy label, one snapshot; next step is a controlled refresh experiment.

### 8b. Social-post cut

> Which pages should a content team review first? I built a client-grouped ranker on 30k real search
> pages. A random split said precision@50 = 0.92 — grouping by client said 0.60. The gap is
> memorisation. Honest number: ~3× the base rate, decision-support only.
> [link] #ML #SEO #Research

### 8c. Employer-facing 3-sentence summary

> I built a leakage-audited, client-grouped ranking model over a 12,023-page slice of a 30,000-page
> production search dataset, comparing a transparent rule against logistic regression, decision trees,
> and a random forest on identical grouped folds. I showed that the evaluation boundary is load-bearing
> — the same model scores 0.92 precision@50 under a random split and 0.60 under honest client-grouping
> — and shipped the output as a human-reviewed action queue with reason codes, a no-go list, and
> monitoring triggers.

## Self-check

- [x] **Every section filled** — markdown thinking AND the code that backs it
- [x] **Runs top to bottom, no errors** — executed with nbconvert, zero error outputs
- [x] **No client names, URLs, or private queries** — anonymised starter slice, aggregate numbers only
- [x] **Careful words** — observed / measured / directional / decision-support; no causal verbs
- [x] **Committed under `work/notebooks/`** — plus the paper in `docs/` and URL in `submission/paper_url.txt`
- [x] **Deployed paper has all 9 sections** — Abstract at the top and Acknowledgments (flyrank.ai) at the bottom
- [x] **ML-12 done** — demo outline + social-post cut + employer-facing 3-sentence summary above

**One-line summary:** a client-grouped, leakage-audited decision-support ranker orders pages for human
review; the honest number is ~3× the base rate, the split gap is the headline lesson, and nothing here
claims that editing a page causally changes its clicks.